In [1]:
import os
os.environ["DEBUG"] = "4"
os.environ["NOOPT"] = "0"

from tinygrad import Tensor

In [2]:
a = Tensor.empty(4, 4)
b = Tensor.empty(4, 4)
(a+b).tolist();

Using LLVM at '/opt/homebrew/opt/llvm/lib/libLLVM.dylib'
opened device METAL from pid:20771
E_4_4
 0: (4, 4)                    float.ptr(16)        (4, 1)
 1: (4, 4)                    float.ptr(16)        (4, 1)
 2: (4, 4)                    float.ptr(16)        (4, 1)
[Opt(op=OptOps.UPCAST, axis=0, arg=4), Opt(op=OptOps.LOCAL, axis=0, arg=4)]
#include <metal_stdlib>
using namespace metal;
kernel void E_4_4(device float* data0, device float* data1, device float* data2, uint3 gid [[threadgroup_position_in_grid]], uint3 lid [[thread_position_in_threadgroup]]) {
  int lidx0 = lid.x; /* 4 */
  int alu0 = (lidx0<<2);
  float4 val0 = *((device float4*)((data1+alu0)));
  float4 val1 = *((device float4*)((data2+alu0)));
  *((device float4*)((data0+alu0))) = float4((val0.x+val1.x),(val0.y+val1.y),(val0.z+val1.z),(val0.w+val1.w));
}
*** METAL      1 E_4_4                                     arg  3 mem  0.00 GB tm     13.75us/     0.01ms (     0.00 GFLOPS    0.0|0.0     GB/s) ['tolist', '__add_

In [3]:
c = Tensor.empty(100, 100)
d = Tensor.empty(100, 100)
(c+d).tolist();

E_625_4_4
 0: (625, 4, 4)               float.ptr(10000)     (16, 4, 1)
 1: (625, 4, 4)               float.ptr(10000)     (16, 4, 1)
 2: (625, 4, 4)               float.ptr(10000)     (16, 4, 1)
[Opt(op=OptOps.UPCAST, axis=0, arg=4), Opt(op=OptOps.LOCAL, axis=0, arg=4)]
#include <metal_stdlib>
using namespace metal;
kernel void E_625_4_4(device float* data0, device float* data1, device float* data2, uint3 gid [[threadgroup_position_in_grid]], uint3 lid [[thread_position_in_threadgroup]]) {
  int gidx0 = gid.x; /* 625 */
  int lidx0 = lid.x; /* 4 */
  int alu0 = ((gidx0<<4)+(lidx0<<2));
  float4 val0 = *((device float4*)((data1+alu0)));
  float4 val1 = *((device float4*)((data2+alu0)));
  *((device float4*)((data0+alu0))) = float4((val0.x+val1.x),(val0.y+val1.y),(val0.z+val1.z),(val0.w+val1.w));
}
*** METAL      3 E_625_4_4                                 arg  3 mem  0.00 GB tm     10.42us/     0.10ms (     0.96 GFLOPS   11.5|11.5    GB/s) ['tolist', '__add__', 'empty']
*** CPU        

In [4]:
from tinygrad.renderer.cstyle import MetalRenderer
from tinygrad.ops import UOp, Ops
from tinygrad import dtypes

const = UOp(Ops.CONST, dtypes.float, arg=1.0)
add = UOp(Ops.ADD, dtypes.float, src=(const, const), arg=None)

print(add)
print("----")
print(MetalRenderer().render([const, add]))

UOp(Ops.ADD, dtypes.float, arg=None, src=(
  x0:=UOp(Ops.CONST, dtypes.float, arg=1.0, src=()),
   x0,))
----
#include <metal_stdlib>
using namespace metal;
kernel void test(uint3 gid [[threadgroup_position_in_grid]], uint3 lid [[thread_position_in_threadgroup]]) {
  float alu0 = (1.0f+1.0f);
}


In [5]:
# load
ptr0 = UOp(Ops.DEFINE_GLOBAL, dtypes.float.ptr(), arg=0)
ptr1 = UOp(Ops.DEFINE_GLOBAL, dtypes.float.ptr(), arg=1)
ptr2 = UOp(Ops.DEFINE_GLOBAL, dtypes.float.ptr(), arg=2)

# blocks (1 thread per)
b0 = UOp(Ops.SPECIAL, dtypes.int, arg=("gidx0", 16))
b1 = UOp(Ops.SPECIAL, dtypes.int, arg=("gidx1", 16))

# load 
off0 = UOp(Ops.ADD, dtypes.long, src=(ptr0, b0))
off1 = UOp(Ops.ADD, dtypes.long, src=(ptr1, b1))
x = UOp(Ops.LOAD, dtypes.float, src=(off0,))
y = UOp(Ops.LOAD, dtypes.float, src=(off1,))

# add
add = UOp(Ops.ADD, dtypes.float, src=(x, y))

# store
stride = UOp(Ops.CONST, dtypes.long, arg=16)
off0_strided = UOp(Ops.MUL, dtypes.long, src=(off0, stride))
no_ptr_off2 = UOp(Ops.ADD, dtypes.long, src=(off0_strided, off1))
off2 = UOp(Ops.ADD, dtypes.long, src=(ptr2, no_ptr_off2))
store = UOp(Ops.STORE, dtypes.void, src=(off2, add))

kernel = MetalRenderer().render([
    ptr0, ptr1, ptr2,
    b0, b1,
    off0, off1,
    x, y,
    add,
    stride, off0_strided, no_ptr_off2, off2,
    store,
])
print(kernel)

#include <metal_stdlib>
using namespace metal;
kernel void test(device float* data0, device float* data1, device float* data2, uint3 gid [[threadgroup_position_in_grid]], uint3 lid [[thread_position_in_threadgroup]]) {
  int gidx0 = gid.x; /* 16 */
  int gidx1 = gid.y; /* 16 */
  long alu0 = (data0+gidx0);
  long alu1 = (data1+gidx1);
  float val0 = *alu0;
  float val1 = *alu1;
  *(data2+(alu0*16ll)+alu1) = (val0+val1);
}


In [6]:
from tinygrad.shape.view import View
a = View.create(shape=(3,2), strides=(2,1))
a = a.permute((1, 0))
print(a.shape)
print(a.strides)
a = a.reshape((3,2))
print(a)


(2, 3)
(1, 2)
None


In [7]:
2**16

65536

In [8]:
a = Tensor.empty(32, 2**16)
b = Tensor.empty(2**16, 128)
(a@b).tolist();

split 256: (32, 128, 65536) -> (32, 128, 256, 256) -> (32, 128, 1)
TENSOR CORES [(1, 128, 1)] [(0, 32, 65536)] WMMA_8_8_8_float_float
r_256_2_2_2_2_2_4_32_2_2_2_2_4_4
 0: (256, 2, 2, 2, 2, 2, 4, 1, 1, 1, 1, 2, 4, 4) float.ptr(1048576)   (1, 512, 32768, 65536, 1024, 131072, 8192, 0, 0, 0, 0, 256, 262144, 2048)
 1: (256, 2, 2, 2, 2, 2, 4, 32, 2, 2, 2, 2, 4, 4) float.ptr(8388608)   (32768, 2, 0, 0, 4, 0, 32, 1024, 128, 256, 512, 1, 0, 8)
 2: (256, 2, 2, 2, 2, 2, 4, 32, 2, 2, 2, 2, 4, 4) float.ptr(2097152)   (256, 0, 65536, 131072, 0, 262144, 0, 8, 1, 2, 4, 0, 524288, 0)
[Opt(op=OptOps.TC, axis=0, arg=(-1, 0)), Opt(op=OptOps.UPCAST, axis=0, arg=4), Opt(op=OptOps.UPCAST, axis=0, arg=4), Opt(op=OptOps.LOCAL, axis=0, arg=4)]
#include <metal_stdlib>
using namespace metal;
float2 __WMMA_8_8_8_float_float(float2 a, float2 b, float2 c){
  simdgroup_float8x8 mat_a, mat_b; simdgroup_float8x8 mat_c;
  mat_a.thread_elements()[0] = a[0]; mat_b.thread_elements()[0] = b[0]; mat_c.thread_elements()[0] = 